In [1]:
# STEP 1: Install dependencies
!pip install tensorflow keras kaggle matplotlib seaborn scikit-learn


In [2]:
from google.colab import files
files.upload()  # choose kaggle.json from your computer


Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"luvinson","key":"3b1e057247978405adc96a758f96a806"}'}

In [3]:
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


In [4]:
!kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset -p ./data --unzip


Dataset URL: https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset
License(s): CC0-1.0
 59% 88.0M/149M [00:00<00:00, 919MB/s]
100% 149M/149M [00:00<00:00, 781MB/s] 


In [5]:
import os

train_dir = "./data/Training"
test_dir = "./data/Testing"

print("Training classes:", os.listdir(train_dir))
print("Testing classes:", os.listdir(test_dir))



Training classes: ['meningioma', 'notumor', 'pituitary', 'glioma']
Testing classes: ['meningioma', 'notumor', 'pituitary', 'glioma']


In [7]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Create two datagens for train & test
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    validation_split=0.15   # use part of training data for validation
)

test_datagen = ImageDataGenerator(rescale=1./255)

# Training set
train_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

# Validation set
val_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

# Test set
test_gen = test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)


Found 4857 images belonging to 4 classes.
Found 855 images belonging to 4 classes.
Found 1311 images belonging to 4 classes.


In [11]:
from tensorflow.keras import layers, models

def build_cnn(input_shape=(224,224,3), num_classes=4):
    model = models.Sequential([
        layers.Conv2D(32, (3,3), activation='relu', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2,2),

        layers.Conv2D(64, (3,3), activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2,2),

        layers.Conv2D(128, (3,3), activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2,2),

        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

cnn_model = build_cnn(num_classes=train_gen.num_classes)
cnn_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
cnn_model.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 222, 222, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 109, 109, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 52, 52, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │    22,151,424 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 4)              │         1,028 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,246,596 (84.86 MB)

 Trainable params: 22,246,148 (84.86 MB)

 Non-trainable params: 448 (1.75 KB)

In [12]:
cnn_history = cnn_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=20
)


Epoch 1/20
152/152 ━━━━━━━━━━━━━━━━━━━━ 1022s 7s/step - accuracy: 0.5049 - loss: 7.8652 - val_accuracy: 0.2713 - val_loss: 20.0735
Epoch 2/20
152/152 ━━━━━━━━━━━━━━━━━━━━ 1050s 7s/step - accuracy: 0.5471 - loss: 1.2808 - val_accuracy: 0.2632 - val_loss: 2.4753
Epoch 3/20
152/152 ━━━━━━━━━━━━━━━━━━━━ 1063s 7s/step - accuracy: 0.5673 - loss: 0.9885 - val_accuracy: 0.5427 - val_loss: 1.9570
Epoch 4/20
152/152 ━━━━━━━━━━━━━━━━━━━━ 1022s 7s/step - accuracy: 0.5969 - loss: 0.9615 - val_accuracy: 0.6398 - val_loss: 1.1353
Epoch 5/20
152/152 ━━━━━━━━━━━━━━━━━━━━ 1009s 7s/step - accuracy: 0.6308 - loss: 0.8597 - val_accuracy: 0.4737 - val_loss: 1.5089
Epoch 6/20
152/152 ━━━━━━━━━━━━━━━━━━━━ 1053s 7s/step - accuracy: 0.6401 - loss: 0.8926 - val_accuracy: 0.6924 - val_loss: 1.2233
Epoch 7/20
152/152 ━━━━━━━━━━━━━━━━━━━━ 1024s 7s/step - accuracy: 0.6300 - loss: 0.9209 - val_accuracy: 0.6924 - val_loss: 1.0322
Epoch 8/20
152/152 ━━━━━━━━━━━━━━━━━━━━ 1003s 7s/step - accuracy: 0.6529 - loss: 0.8422 -

In [16]:
cnn_model.save("brain_tumor_cnn_model.h5")



In [17]:
from google.colab import files
files.download("brain_tumor_cnn_model.h5")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>